# Fiche — Quel algorithme RL choisir ?

Il existe beaucoup d'algorithmes RL. Cette fiche explique les différences clés et comment choisir.

---
## La question la plus importante : quel type d'action ?

Avant tout, demande-toi : **qu'est-ce que l'agent peut faire ?**

### Action Discrète
L'agent choisit parmi une **liste finie** d'options. Comme appuyer sur un bouton.

```
Exemples : gauche / droite / haut / bas
           sauter / courir / attaquer
           acheter / vendre / tenir
```

→ En code : `spaces.Discrete(4)` → entiers 0, 1, 2, 3

### Action Continue
L'agent choisit une **valeur dans une plage**. Comme tourner un volant.

```
Exemples : force du moteur entre -1.0 et +1.0
           angle de rotation entre -180° et 180°
           quantité d'insuline entre 0 et 100 unités
```

→ En code : `spaces.Box(low=-1, high=1, shape=(2,))`

In [ ]:
import gymnasium as gym

# Exemples d'espaces d'action
env_discret = gym.make('LunarLander-v3')       # discret
env_continu = gym.make('LunarLanderContinuous-v3')  # continu

print("LunarLander (discret)  :", env_discret.action_space)
print("LunarLander (continu)  :", env_continu.action_space)

env_discret.close()
env_continu.close()

---
## Les algorithmes disponibles dans Stable-Baselines3

| Algorithme | Action Discrète | Action Continue | Remarque |
|-----------|:-:|:-:|----------|
| **PPO** | ✅ | ✅ | Meilleur point de départ, robuste |
| **A2C** | ✅ | ✅ | Plus simple que PPO, plus volatile |
| **DQN** | ✅ | ❌ | Discret uniquement |
| **SAC** | ❌ | ✅ | Très efficace en continu |
| **TD3** | ❌ | ✅ | Variante de DDPG, plus stable |
| **DDPG** | ❌ | ✅ | Ancêtre de TD3, moins utilisé |

> **Règle simple** : commence toujours par **PPO**. Il marche dans la majorité des cas.

---
## Comprendre les grandes familles

### On-policy vs Off-policy

C'est la distinction technique la plus importante.

**On-policy** (PPO, A2C)
- L'agent apprend **uniquement** de ce qu'il vient de faire
- Chaque expérience est utilisée une fois, puis jetée
- Analogie : un élève qui révise **seulement** les exercices du cours d'aujourd'hui
- Avantage : plus stable, plus fiable
- Inconvénient : moins efficace en données (besoin de beaucoup de steps)

**Off-policy** (SAC, TD3, DQN, DDPG)
- L'agent stocke ses expériences passées dans un **replay buffer**
- Il peut **réapprendre** des expériences anciennes
- Analogie : un élève qui garde toutes ses fiches et révise n'importe quelle leçon passée
- Avantage : apprend plus vite (moins de steps nécessaires)
- Inconvénient : peut être instable, plus complexe à régler

```
On-policy  :  [expérience] → [apprendre] → [jeter]
Off-policy :  [expérience] → [stocker dans buffer] → [piocher et apprendre]
```

---
## Fiche détaillée par algorithme

### PPO — Proximal Policy Optimization
- **Famille** : On-policy
- **Actions** : Discrètes ET continues
- **Idée** : Améliore la politique par petits pas pour éviter de trop s'éloigner de la version précédente ("proximal" = proche)
- **Quand l'utiliser** : Toujours en premier. Bon équilibre stabilité / performance.
- **Environnements** : LunarLander, MuJoCo, Snake custom, jeux Atari

```python
from stable_baselines3 import PPO
model = PPO('MlpPolicy', env, verbose=1)
```

---

### A2C — Advantage Actor-Critic
- **Famille** : On-policy
- **Actions** : Discrètes ET continues
- **Idée** : Deux réseaux : un **Actor** (choisit l'action) et un **Critic** (évalue si c'était une bonne idée)
- **Quand l'utiliser** : Alternative à PPO, utile pour comparer. Peut converger plus vite sur certains problèmes.
- **Limite** : Plus volatile que PPO (les courbes de reward varient plus)

```python
from stable_baselines3 import A2C
model = A2C('MlpPolicy', env, verbose=1)
```

---

### DQN — Deep Q-Network
- **Famille** : Off-policy
- **Actions** : Discrètes uniquement
- **Idée** : Apprend la "valeur" de chaque action possible, choisit la meilleure
- **Quand l'utiliser** : Quand l'espace d'action est petit et discret. Jeux Atari classiques.
- **Limite** : Ne fonctionne pas avec des actions continues

```python
from stable_baselines3 import DQN
model = DQN('MlpPolicy', env, verbose=1)
```

---

### SAC — Soft Actor-Critic
- **Famille** : Off-policy
- **Actions** : Continues uniquement
- **Idée** : Maximise la récompense ET l'**entropie** (exploration). "Soft" = garde une certaine randomité.
- **Quand l'utiliser** : Robustique, contrôle de bras robotique, systèmes physiques. Très efficace en échantillons.
- **Avantage** : Souvent le meilleur en continu

```python
from stable_baselines3 import SAC
model = SAC('MlpPolicy', env, verbose=1)  # env doit avoir Box action space
```

---

### TD3 — Twin Delayed DDPG
- **Famille** : Off-policy
- **Actions** : Continues uniquement
- **Idée** : Corrige les problèmes de DDPG avec deux Critic réseaux ("Twin") et des mises à jour décalées ("Delayed")
- **Quand l'utiliser** : Alternative à SAC en continu. Légèrement plus déterministe.

```python
from stable_baselines3 import TD3
model = TD3('MlpPolicy', env, verbose=1)
```

---
## Arbre de décision : lequel choisir ?

```
Mon espace d'action est...
│
├── DISCRET (liste d'options)
│   │
│   ├── Je commence / je veux quelque chose de simple
│   │   └── → PPO
│   │
│   ├── Je veux comparer un autre algo
│   │   └── → A2C
│   │
│   └── Petit espace d'action, jeu Atari-like
│       └── → DQN
│
└── CONTINU (valeurs dans une plage)
    │
    ├── Je commence / je veux quelque chose de robuste
    │   └── → PPO (marche aussi en continu)
    │
    ├── Robotique, physique, je veux la meilleure perf
    │   └── → SAC
    │
    └── Je veux un algo plus déterministe que SAC
        └── → TD3
```

---
## Les deux types de Policy (réseau de neurones)

SB3 te demande toujours de choisir une **Policy**, c'est-à-dire l'architecture du réseau de neurones.

| Policy | Quand l'utiliser |
|--------|------------------|
| `MlpPolicy` | Observations sous forme de **vecteur** (liste de nombres). Le cas le plus courant. |
| `CnnPolicy` | Observations sous forme d'**image** (pixels). Jeux Atari, vision par ordinateur. |
| `MultiInputPolicy` | Observations **mixtes** (vecteur + image en même temps). |

Pour LunarLander (8 valeurs numériques) → `MlpPolicy`  
Pour Atari Breakout (image 84×84) → `CnnPolicy`

---
## Exemple : tester plusieurs algos sur le même environnement

In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO, A2C, DQN
from stable_baselines3.common.evaluation import evaluate_policy

TIMESTEPS = 50_000

algos = {
    "PPO": PPO,
    "A2C": A2C,
    "DQN": DQN,
}

results = {}

for name, AlgoClass in algos.items():
    env = gym.make('LunarLander-v3')
    model = AlgoClass('MlpPolicy', env, verbose=0)
    model.learn(total_timesteps=TIMESTEPS)
    mean, std = evaluate_policy(model, env, n_eval_episodes=20)
    results[name] = (mean, std)
    env.close()
    print(f"{name:<5} → reward moyen : {mean:>8.2f} ± {std:.2f}")

print()
best = max(results, key=lambda k: results[k][0])
print(f"Meilleur algo sur LunarLander ({TIMESTEPS} steps) : {best}")

---
## Résumé en une image mentale

```
┌─────────────────────────────────────────────────┐
│              ALGORITHMES RL (SB3)               │
├──────────────────────┬──────────────────────────┤
│     ON-POLICY        │      OFF-POLICY           │
│  (apprend du présent)│  (apprend du passé aussi) │
│                      │                           │
│  PPO  ← le meilleur  │  SAC  ← meilleur continu  │
│  A2C  ← plus simple  │  TD3  ← alternatif SAC    │
│                      │  DQN  ← discret seulement │
├──────────────────────┴──────────────────────────┤
│  Discret : PPO, A2C, DQN                        │
│  Continu : PPO, A2C, SAC, TD3, DDPG             │
│  Les deux : PPO, A2C                            │
│                                                 │
│  → Toujours commencer par PPO                   │
└─────────────────────────────────────────────────┘
```

**Une seule règle à retenir** : commence par PPO, change seulement si tu as une raison précise.

---
## Hyperparamètres importants

Quand un modèle n'apprend pas bien, on peut régler ces paramètres (par ordre de priorité) :

| Paramètre | Défaut PPO | Rôle |
|-----------|-----------|------|
| `learning_rate` | 3e-4 | Vitesse d'apprentissage. Trop élevé = instable. Trop faible = lent. |
| `n_steps` | 2048 | Steps collectés avant chaque mise à jour |
| `batch_size` | 64 | Taille des mini-lots pour l'entraînement |
| `n_epochs` | 10 | Nombre de passes sur les données collectées |
| `gamma` | 0.99 | Facteur d'actualisation (importance du futur). 1 = très lointain, 0 = maintenant seulement |
| `ent_coef` | 0.0 | Coefficient d'entropie (encourage l'exploration) |

**Conseil débutant** : ne touche pas aux hyperparamètres avant d'avoir suffisamment de timesteps d'entraînement.

In [ ]:
# Exemple : PPO avec hyperparamètres personnalisés
from stable_baselines3 import PPO
import gymnasium as gym

env = gym.make('LunarLander-v3')

model = PPO(
    'MlpPolicy',
    env,
    verbose=1,
    learning_rate=1e-4,   # plus lent mais plus stable
    n_steps=1024,          # moins de steps par update
    gamma=0.995,           # accorde plus d'importance au futur
    ent_coef=0.01,         # encourage un peu plus l'exploration
)

print("Modèle créé avec hyperparamètres custom")
print(f"learning_rate : {model.learning_rate}")
print(f"gamma         : {model.gamma}")
print(f"n_steps       : {model.n_steps}")
env.close()